In [ ]:
!pip install -q sentence-transformers faiss-cpu scikit-learn python-docx anthropic pandas

In [ ]:
import os,re,math,json,glob,pickle,logging,warnings
from dataclasses import dataclass,field,asdict
from typing import Any, Callable, Dict,List, Optional
from collections import Counter,defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO,format="%(levelname)s|%(message)s",force=True)
for noisy in("https","sentence_transformers","urllib3","huggingface_hub","faiss","faiss.loader"):
  logging.getLogger(noisy).setLevel(logging.WARNING)
log=logging.getLogger("rag")
"""
llm_backend:str="auto"
ollama_model:str="qwen2.5:7b"
ollama_url:str="http://localhost:11434"
hf_model:str="Qwen/Qwen2.5-1.5B-Instruct"
llm_temperature:float=0.1
"""

@dataclass
class RAGConfig:
  chunk_size:int=120
  chunk_overlap:int=25
  add_title_to_chunk:bool=True

  embed_model:str="infloat/multilingual-e5-small"
  rerank_model:str="cross-encode/mmarco-mMiniLMv2-l12-H384-v1"
  llm_model:str="api key"
  k_dense: int=20 #dense anlamsal bag semantık olarak bağ olan
  k_sparse:int=20 #spark ıse kelımenın kokune bakıyor
  rrf_k:int=60
  dense_weight:float=1.0
  sparse_weight:float=1.0
  k_candidates:int=12 #tum yapıdan benzer kac tane getırecegım
  k_final:int=4. #12 ornegın kacını rerank e sokacagız.
  mmr_lambda:float=0.75 #anlamsal butunluk saglayabılmek ıcın kelıme sayısının kacının benzer olması gerektıgını belırtıyoruz.
  neightbor_window:int=1 #neıghbor wındow degerı de arttırılınca genellemıs oluyoruz.

  use_condense:bool=True #sohbet gecmısındekı verılerıde kullanır
  use_multi_query:bool=True #llm ler sorumuzuda bir kaç şekle sorar ve daha fazla arama yapmak ıcın kullanırız
  n_multi_query:int=3  #kac farklı sekılde sorular soracagını
  use_hyde:bool=False # egerkı bu false olmazsa baglamın uzerıne kendı yorumlarını vs de sokar varsayımsal da çıkarım yapar true olursa
  use_rerank:bool=True

cfg=RAGConfig()
cfg

RAGConfig(chunk_size=120, chunk_overlap=25, add_title_to_chunk=True, embed_model='infloat/multilingual-e5-small', rerank_model='cross-encode/mmarco-mMiniLMv2-l12-H384-v1', llm_model='api key', k_dense=20, k_sparse=20, rrf_k=60, dense_weight=1.0, sparse_weight=1.0, k_candidates=12, k_final=4.0, mmr_lambda=0.75, neightbor_window=1, use_condense=True, use_multi_query=True, n_multi_query=3, use_hyde=False, use_rerank=True)

In [ ]:
@dataclass
class Document:
  text:str
  metadata:Dict[str,Any]=field(default_factory=dict)

def clean_text(text:str)->str:
  text=text.replace("\r","")
  text =re.sub(r"(\w)"-"\n(\w)",r"\1\2",text)
  text = re.sub(r"[ \t]+","",text)
  text=re.sub(r"\n{3,}","\n\n",text)
  return text.strip()

def load_file(path:str)->List[Document]:
  """TXT,MD,PDF ve DOCX dosyalarını okur. PDF'te her sayfa ayrı belge olur (sayfa numarası metadata'da)."""
  ext=os.path.splitext(path)[1].lower()
  base={"source":os.path.basename(path),"title":os.path.splitext(os.path.basename(path))[0]}
  if ext in (".txt",".md"):
    with open(path,encoding="utf-8") as f:
      return [Document(text=clean_text(f.read()),metadata={**base,"source":path})[0]]
  if ext == ".pdf":
    from pypdf import PdfReader
    reader=PdfReader(path)
    return [Document(clean_text(p.extract_text() or "")),{**base,"page":i+1}]
  if ext==".docx":
    import docx
    d=docx.Document(path)
    return [Document(clean_text("\n\n".join(p.text for p in d.paragraphs if p.text.strip()))),base]
  raise ValueError(f"Desteklenmeyen Dosya Türü: {ext}")

def load_directory(path:str,exts=(".txt",".md",".pdf",".docx"),extra_metadata:Optional[dict]=None) -> List[Document]:
  docs=[]
  for f in sorted(glob.glob(os.path.join(path,"**","*"),recursive=True)):
    if f.lower().endswith(exts):
      try:
        for d in load_file(f):
          d.metadata.update(extra_metadata or {})
          docs.append(d)
      except Exception as e:
        log.warning(f"{f} okunamadı: {e}")
  log.info(f"{len(docs)} belge yüklendi({path})")
  return docs





In [ ]:
documents = [
    Document("""Yıllık İzin Politikası

Çalışanlar kıdemlerine göre yıllık ücretli izin hakkı kazanır. Bir yıldan beş yıla kadar (beş yıl dahil) kıdemi olan çalışanlara yılda 14 iş günü izin verilir. Beş yıldan fazla, on beş yıldan az kıdemi olanlara 20 iş günü, on beş yıl ve üzeri kıdemi olanlara 26 iş günü izin verilir. Elli yaş ve üzeri çalışanlar kıdemlerinden bağımsız olarak en az 20 iş günü izin kullanır.

İzin talepleri en az iki hafta önceden İKSİS portalı üzerinden yapılır ve doğrudan yönetici tarafından onaylanır. Beş iş gününden uzun izinler için ayrıca departman direktörünün onayı gerekir.

Kullanılmayan izinlerin en fazla 5 günü bir sonraki yıla devredilebilir. Devreden izinler 31 Mart tarihine kadar kullanılmazsa yanar.

Babalık izni 10 iş günüdür. Hastalık raporları, rapor başlangıcından itibaren 3 iş günü içinde İnsan Kaynakları birimine iletilmelidir.""",
             {"source": "izin_politikasi.md", "title": "Yıllık İzin Politikası", "category": "ik", "year": 2025}),

    Document("""Uzaktan ve Hibrit Çalışma Politikası

Ofis dışı çalışmaya uygun pozisyonlarda çalışanlar haftada en fazla 3 gün uzaktan çalışabilir. Salı ve perşembe günleri tüm ekipler için ortak ofis günüdür.

Uzaktan çalışma günlerinde 10:00 ile 16:00 arası çekirdek çalışma saatidir; bu saatlerde çalışanın ulaşılabilir olması beklenir. Şirket sistemlerine yalnızca kurumsal VPN üzerinden bağlanılabilir.

Uzaktan çalışan personele aylık 750 TL internet ve ev ofis desteği ödenir. Ergonomik sandalye ve monitör talepleri İK portalından yapılır.

Yurt dışından uzaktan çalışma yılda en fazla 30 takvim günü ile sınırlıdır ve yönetici ile İK onayı gerektirir.""",
             {"source": "uzaktan_calisma.md", "title": "Uzaktan Çalışma Politikası", "category": "ik", "year": 2024}),

    Document("""Seyahat ve Masraf Politikası

İş amaçlı tüm masraflar, harcama tarihinden itibaren en geç 30 gün içinde masraf sistemine girilmelidir. Her masraf kalemi için fatura veya fiş yüklenmesi zorunludur; belgesiz masraflar ödenmez.

Şehir içi yemek masraflarında kişi başı günlük limit 600 TL, şehir dışı seyahatlerde 900 TL'dir. Konaklamada dört yıldız üzeri oteller için önceden finans onayı alınmalıdır.

Taksi masrafları yalnızca 22:00 sonrası veya toplu taşımanın mümkün olmadığı durumlarda karşılanır.

Onay akışı: 5.000 TL'ye kadar olan masrafları doğrudan yönetici onaylar. 5.000 TL üzerindeki masraflar için ek olarak finans direktörünün onayı gerekir. Onaylanan masraflar bir sonraki maaş ödemesiyle birlikte hesaba yatırılır.""",
             {"source": "masraf_politikasi.md", "title": "Seyahat ve Masraf Politikası", "category": "finans", "year": 2025}),

    Document("""Bilgi Güvenliği Politikası (2025)

Tüm kullanıcı parolaları en az 12 karakter uzunluğunda olmalı; büyük harf, küçük harf, rakam ve özel karakter içermelidir. Parolalar 90 günde bir değiştirilir. Tüm kurumsal hesaplarda çok faktörlü kimlik doğrulama (MFA) zorunludur.

Kişisel USB bellek ve harici disklerin şirket bilgisayarlarına takılması yasaktır. Dosya paylaşımı yalnızca kurumsal bulut depolama üzerinden yapılır.

Şüpheli e-postalar (oltalama / phishing) açılmadan guvenlik@ornekteknoloji.com adresine iletilmelidir. Şirket dizüstü bilgisayarının veya telefonunun kaybolması ya da çalınması durumunda en geç 1 saat içinde BT Yardım Masasına bildirim yapılmalıdır.

Veriler dört sınıfa ayrılır: Genel, Kurum İçi, Gizli ve Çok Gizli. Gizli ve Çok Gizli veriler şifrelenmeden e-posta ile gönderilemez.""",
             {"source": "bilgi_guvenligi_2025.md", "title": "Bilgi Güvenliği Politikası", "category": "it", "year": 2025}),

    Document("""Bilgi Güvenliği Politikası (2022 — yürürlükten kaldırıldı)

Kullanıcı parolaları en az 8 karakter olmalıdır. Parolalar 60 günde bir değiştirilir. Çok faktörlü kimlik doğrulama yalnızca yönetici hesapları için zorunludur.

Şüpheli e-postalar BT Yardım Masasına telefonla bildirilir. Cihaz kayıpları 24 saat içinde bildirilmelidir.""",
             {"source": "bilgi_guvenligi_2022.md", "title": "Bilgi Güvenliği Politikası (eski)", "category": "it", "year": 2022}),

    Document("""Eğitim ve Gelişim Politikası

Her çalışanın yıllık kişisel eğitim bütçesi 15.000 TL'dir. Bütçe; kurslar, konferanslar, kitaplar ve çevrim içi eğitim platformları için kullanılabilir. Kullanılmayan bütçe bir sonraki yıla devretmez.

Pozisyonla ilgili profesyonel sertifika sınav ücretleri, ilk denemede başarısız olunsa bile bir kez daha şirket tarafından karşılanır.

Tutarı 30.000 TL'yi aşan eğitimler için eğitim taahhütnamesi imzalanır. Çalışan eğitim bitiminden sonraki 12 ay içinde kendi isteğiyle ayrılırsa eğitim bedelinin kalan süreye düşen kısmını iade eder.

Yeni çalışanlar ilk 30 gün içinde zorunlu oryantasyon ve bilgi güvenliği eğitimlerini tamamlamalıdır.""",
             {"source": "egitim_gelisim.md", "title": "Eğitim ve Gelişim Politikası", "category": "ik", "year": 2025}),
]
print(len(documents), "belge")
pd.DataFrame([d.metadata for d in documents])

6 belge


,source,title,category,year
0,izin_politikasi.md,Yıllık İzin Politikası,ik,2025
1,uzaktan_calisma.md,Uzaktan Çalışma Politikası,ik,2024
2,masraf_politikasi.md,Seyahat ve Masraf Politikası,finans,2025
3,bilgi_guvenligi_2025.md,Bilgi Güvenliği Politikası,it,2025
4,bilgi_guvenligi_2022.md,Bilgi Güvenliği Politikası (eski),it,2022
5,egitim_gelisim.md,Eğitim ve Gelişim Politikası,ik,2025


In [ ]:
@dataclass
class Chunk:
    id: int
    text: str              # LLM'e gösterilecek metin
    embed_text: str        # vektörleştirilecek metin (başlık eklenmiş)
    doc_id: str
    position: int          # belge içindeki sıra (komşu genişletme için)
    metadata: Dict[str, Any]


_SENT_SPLIT = re.compile(r"(?<=[.!?…])\s+(?=[A-ZÇĞİÖŞÜ0-9\"“(])")

def split_sentences(text: str) -> List[str]:
    sents = []
    for para in re.split(r"\n\s*\n", text):
        para = re.sub(r"\s+", " ", para).strip()
        if para:
            sents.extend(s.strip() for s in _SENT_SPLIT.split(para) if s.strip())
    return sents


def chunk_text(text: str, size: int, overlap: int) -> List[str]:
    # Hedef boydan uzun cümleleri kelime pencerelerine böl
    units = []
    for s in split_sentences(text):
        w = s.split()
        if len(w) > size:
            step = max(1, size - overlap)
            units.extend(" ".join(w[i:i + size]) for i in range(0, len(w), step))
        else:
            units.append(s)

    chunks, cur, cur_len = [], [], 0
    for u in units:
        n = len(u.split())
        if cur and cur_len + n > size:
            chunks.append("\n".join(cur))
            # Örtüşme: son cümleleri bir sonraki parçaya taşı
            keep, kl = [], 0
            for t in reversed(cur):
                tl = len(t.split())
                if kl + tl > overlap:
                    break
                keep.insert(0, t); kl += tl
            cur, cur_len = keep, kl
        cur.append(u); cur_len += n
    if cur:
        chunks.append("\n".join(cur))
    return chunks          # parça içinde her satır bir cümle


def make_chunks(docs: List[Document], cfg: RAGConfig, start_id: int = 0) -> List[Chunk]:
    out, cid = [], start_id
    for d in docs:
        doc_id = d.metadata.get("source", "doc") + (f"#s{d.metadata['page']}" if "page" in d.metadata else "")
        title = d.metadata.get("title", "")
        for pos, t in enumerate(chunk_text(d.text, cfg.chunk_size, cfg.chunk_overlap)):
            et = f"{title}: {t}" if (cfg.add_title_to_chunk and title) else t
            out.append(Chunk(cid, t, et, doc_id, pos, dict(d.metadata)))
            cid += 1
    return out


# Demo için küçük parçalar: örnek belgeler kısa olduğu için
cfg.chunk_size, cfg.chunk_overlap = 20, 12
_demo = make_chunks(documents, cfg)
print(f"{len(_demo)} parça | ortalama {np.mean([len(c.text.split()) for c in _demo]):.0f} kelime\n")
for c in _demo[:5]:
    print(f"[{c.id}] {c.doc_id} / {c.position}: {c.text[:110].replace(chr(10), ' ')}...")

45 parça | ortalama 18 kelime

[0] izin_politikasi.md / 0: Yıllık İzin Politikası Çalışanlar kıdemlerine göre yıllık ücretli izin hakkı kazanır....
[1] izin_politikasi.md / 1: Yıllık İzin Politikası Çalışanlar kıdemlerine göre yıllık ücretli izin hakkı kazanır. Bir yıldan beş yıla kada...
[2] izin_politikasi.md / 2: Beş yıldan fazla, on beş yıldan az kıdemi olanlara 20 iş günü, on beş yıl ve üzeri kıdemi olanlara 26...
[3] izin_politikasi.md / 3: olanlara 20 iş günü, on beş yıl ve üzeri kıdemi olanlara 26 iş günü izin verilir....
[4] izin_politikasi.md / 4: üzeri kıdemi olanlara 26 iş günü izin verilir....


In [ ]:
TR_STOP = set("""acaba ama ancak artık aslında az bana bazen bazı belki ben beni benim bir biri birkaç birçok biz bu buna
bunda bundan bunu bunun burada çok çünkü da daha de defa diye eğer en gibi hem hep hepsi her hiç için ile ise kaç kez ki kim
mi mu mü mı nasıl ne neden nerede niçin o olan olarak oldu olduğu olmak olur ona ondan onlar onu onun öyle sanki şey siz şu
tüm ve veya ya yani hangi peki var yok""".split())

def tr_lower(s: str) -> str:
    return s.replace("I", "ı").replace("İ", "i").lower()

def tokenize(text: str, stem: int = 5) -> List[str]:
    toks = re.findall(r"[0-9a-zçğıöşüâîû]+", tr_lower(text))
    return [t[:stem] if (stem and not t.isdigit()) else t for t in toks if t not in TR_STOP and len(t) > 1]


class BM25:
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1, selfb = k1, b

    def fit(self, corpus_tokens: List[List[str]]):
        self.N = len(corpus_tokens)
        self.doc_len = np.array([len(d) for d in corpus_tokens], dtype=float)
        self.avgdl = self.doc_len.mean() if self.N else 1.0
        df = Counter(t for d in corpus_tokens for t in set(d))
        self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in df.items()}
        self.inverted = defaultdict(list)                      # terim → [(parça, frekans)]
        for i, d in enumerate(corpus_tokens):
            for t, f in Counter(d).items():
                self.inverted[t].append((i, f))
        return self

    def scores(self, query_tokens: List[str]) -> np.ndarray:
      S=np.zeros(self.N)
      for t in (query_tokens):
        if idf is None:
          continue
        for i, f in self.inverted[t]:
          S[i]+=idf*f*(slef.k1+1)/(f+self.k1)*(1-self.b+self.b*self.doc_len[i]/self.avgdl)
          return s
print(tokenize("İzin taleplerinin İs-Ha portalından yapılması zorunludur."))


['izin', 'talep', 'is', 'ha', 'porta', 'yapıl', 'zorun']


In [ ]:
def reciprocal_rank_fusion(ranked_lists: List[List[int]], k: int = 60, weights: Optional[List[float]] = None) -> List[tuple]:
    scores = defaultdict(float)
    weights = weights or [1.0] * len(ranked_lists)
    for w, lst in zip(weights, ranked_lists):
        for rank, idx in enumerate(lst):
            scores[idx] += w / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])


class Reranker:
    def __init__(self, model_name: str):
        try:
            from sentence_transformers import CrossEncoder
            self.model = CrossEncoder(model_name, max_length=512)
            self.available = True
        except Exception as e:
            log.warning(f"Reranker yüklenemedi ({type(e).__name__}). Reranking atlanacak, RRF sırası kullanılacak.")
            self.model, self.available = None, False

    def score(self, query: str, texts: List[str]) -> np.ndarray:
        return np.asarray(self.model.predict([(query, t) for t in texts], show_progress_bar=False), dtype=float)


def mmr_select(candidates: List[int], relevance: Dict[int, float], vectors: np.ndarray, k: int, lam: float) -> List[int]:
    if not candidates:
        return []
    r = np.array([relevance[i] for i in candidates])
    r = (r - r.min()) / (r.max() - r.min() + 1e-9)          # alaka puanlarını 0–1'e ölçekle
    rel = dict(zip(candidates, r))
    selected, pool = [], list(candidates)
    while pool and len(selected) < k:
        if not selected:
            best = max(pool, key=lambda i: rel[i])
        else:
            S = vectors[selected]
            best = max(pool, key=lambda i: lam * rel[i] - (1 - lam) * float(np.max(S @ vectors[i])))
        selected.append(best)
        pool.remove(best)
    return selected